# Research Agent Receipts — ledger CSV export

Research agent pays for APIs (including x402 URLs) and exports an **expense CSV** from the ledger.

**Prereq:** `pip install -e "./python[dev]" jupyter`

In [ ]:
import csv
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))
from _notebook_utils import FAKE_RECIPIENT, enable_mock_router, display_ledger

from algopay import AlgoPay
from algopay.core.types import Network

X402_URL = "https://example.com/paid-research-api"
CSV_PATH = Path("research_receipts.csv")

client = AlgoPay(network=Network.ALGORAND_TESTNET)
enable_mock_router(client, x402_aware=True)

In [ ]:
ws = client.wallet.create_wallet_set("research-agent")
wallet = client.wallet.create_wallet(ws.id)
await client.add_budget_guard(wallet.id, daily_limit="1.0", name="research_budget")

calls = [
    (FAKE_RECIPIENT, "0.02", "Academic paper metadata lookup"),
    (X402_URL, "0.05", "x402 research API - market signals"),
    (FAKE_RECIPIENT, "0.01", "Citation graph expansion"),
]

for recipient, amount, purpose in calls:
    r = await client.pay(wallet.id, recipient, amount, purpose=purpose)
    tag = "ok" if r.success else r.status.value
    print(f"{tag:8} {amount} USDC -> {str(recipient)[:40]}...")

In [ ]:
entries = await client.ledger.query(wallet_id=wallet.id, limit=500)
display_ledger(entries)

In [ ]:
fieldnames = [
    "timestamp", "wallet_id", "recipient", "amount_usdc",
    "status", "purpose", "tx_hash", "method",
]
rows = [
    {
        "timestamp": e.timestamp.isoformat(),
        "wallet_id": e.wallet_id,
        "recipient": e.recipient,
        "amount_usdc": str(e.amount),
        "status": e.status.value,
        "purpose": e.purpose or "",
        "tx_hash": e.tx_hash or "",
        "method": e.method,
    }
    for e in entries
]

with CSV_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"Exported {len(rows)} rows -> {CSV_PATH.resolve()}")